In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
df = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv("/Volumes/week7/default/week7/Sample - Superstore 4.csv")
)

display(df)

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, count, when
exprs = []
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        exprs.append(
            count(
                when(
                    col(field.name).isNull() | (trim(col(field.name)) == ""),
                    field.name
                )
            ).alias(field.name)
        )
    else:
        exprs.append(
            count(
                when(col(field.name).isNull(), field.name)
            ).alias(field.name)
        )
missing_counts = df.select(exprs)
display(missing_counts)

In [0]:
#Data Cleaning
from pyspark.sql.functions import col
clean_df = df.filter(col("Customer ID").isNotNull())
clean_df = clean_df.dropDuplicates()
print("Row Count after Cleaning:", clean_df.count())

display(clean_df)

In [0]:
# Replace spaces with underscores in all column names
clean_df = clean_df.toDF(*[c.replace(" ", "_") for c in clean_df.columns])

print(clean_df.columns)
display(clean_df)

In [0]:
delta_path = "/Volumes/week7/default/week7/customer_delta"

clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)

print("Delta table created successfully.")

In [0]:
delta_df = spark.read.format("delta").load(delta_path)
display(delta_df)

In [0]:
incremental_df = incremental_df.dropDuplicates(["Order_ID"])

In [0]:
from pyspark.sql.functions import col

incremental_update = (
    clean_df
    .filter(col("Row_ID").isin(1, 2, 3))
    .withColumn("Sales", col("Sales") + lit(500.0))
    .withColumn("Profit", col("Profit") + lit(100.0))
)

display(incremental_update)

In [0]:
incremental_new = (
    clean_df
    .filter(col("Row_ID").isin(4, 5))
    .withColumn("Row_ID", col("Row_ID") + lit(10000))
)

display(incremental_new)

In [0]:
incremental_df = incremental_update.unionByName(incremental_new)
display(incremental_df)

In [0]:
incremental_df = incremental_df.dropDuplicates(["Row_ID"])

In [0]:
from delta.tables import DeltaTable
delta_table = DeltaTable.forPath(spark, delta_path)
(
    delta_table.alias("target")
    .merge(
        incremental_df.alias("source"),
        "target.Row_ID = source.Row_ID"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
print("MERGE completed successfully.")

In [0]:
final_df = spark.read.format("delta").load(delta_path)
display(final_df)

In [0]:
print("Total Records:", final_df.count())

In [0]:
from pyspark.sql.functions import col
duplicates = (
    final_df.groupBy("Row_ID")
            .count()
            .filter(col("count") > 1)
)

duplicates.show()

In [0]:
display(
    final_df.filter(col("Row_ID").isin(1, 2, 3, 10004, 10005))
)